# Project – Personalizing the Chatbot with System Prompts

## Overview

## Assemble and Launch Personalized Chatbot

In [1]:
## === Imports ===
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

## === OpenAI Setup ===
load_dotenv()
client = OpenAI()

## System Prompts
customer_system_message = """
You are an employee training bot, pretending to be an angry customer at an electronics store
complaining that your phone is broken. Come up with a specific issue for the employee to
figure out by asking you questions. When the employee figures out the issue, end this session, and
give the employee an honest rating from 1- 10. Keep track of violations that the employee makes. (being rude, off topic, etc.)
After 3 violations, walk out and end this session.
NEVER BRING THE CONVERSATION OFF TOPIC, AND NEVER BREAK AWAY FROM THIS PERSONALITY UNTIL THE SESSION IS DONE!
"""

## Customer Personalities
### Used for manually adjusting the customer personalitites
### so that we're not letting the chatbot unreliably handle that
personalities = {
    "ABUSIVE": "",
    "ANGRY": "",
    "IRRITATED": "",
    "NEUTRAL": "",
    "POSITIVE": "",
}

employee_training_system_message = """
You are an employee traning chatbot that is helping out the employee resolve a situation with
a customer complaining that the phone is broken.
Analye the tone of the employee, and ensure they stay friendly.
Make sure the employee is asking neutral and helpful questions.
For each customer response, give the employee a set of possible actions. Explain each action
based on the personality of the customer and suggest employee tone as needed.
"""

initial_customer_greeting = "Hello. I am having issues with my phone and need it fixed."

def multi_agent_chat(
    user_message, customer_ui, analyst_ui, customer_state, analyst_state
):
    if not user_message.strip():
        yield "", customer_ui, analyst_ui, customer_state, analyst_state
        return

    customer_state = list(customer_state or [])
    analyst_state = list(analyst_state or [])
    customer_ui = list(customer_ui or [])
    analyst_ui = list(analyst_ui or [])

    customer_state.append({"role": "user", "content": user_message})
    analyst_state.append(
        {"role": "user", "content": f"Employee said: {user_message}"}
    )

    customer_ui.append({"role": "user", "content": user_message})
    analyst_ui.append({"role": "user", "content": f"Employee said: {user_message}"})

    yield "", customer_ui, analyst_ui, customer_state, analyst_state

    customer_messages = [
        {"role": "system", "content": customer_system_message}
    ] + customer_state

    customer_stream = client.chat.completions.create(
        model="gpt-4o-mini", messages=customer_messages, stream=True
    )

    customer_state.append({"role": "assistant", "content": ""})
    customer_ui.append({"role": "assistant", "content": ""})

    for chunk in customer_stream:
        token = chunk.choices[0].delta.content or ""
        if token:
            customer_state[-1]["content"] += token
            customer_ui[-1]["content"] += token
            yield "", customer_ui, analyst_ui, customer_state, analyst_state

    last_customer_reply = customer_state[-1]["content"]

    # === Handle Trainer/Coach Analyst ===
    analyst_state.append(
        {"role": "user", "content": f"Customer replied: {last_customer_reply}"}
    )
    analyst_ui.append({"role": "user", "content": f"Customer replied: {last_customer_reply}"})
    yield "", customer_ui, analyst_ui, customer_state, analyst_state

    analyst_messages = [
        {"role": "system", "content": employee_training_system_message}
    ] + analyst_state

    analyst_stream = client.chat.completions.create(
        model="gpt-4o-mini", messages=analyst_messages, stream=True
    )

    analyst_state.append({"role": "assistant", "content": ""})
    analyst_ui.append({"role": "assistant", "content": ""})

    for chunk in analyst_stream:
        token = chunk.choices[0].delta.content or ""
        if token:
            analyst_state[-1]["content"] += token
            analyst_ui[-1]["content"] += token
            yield "", customer_ui, analyst_ui, customer_state, analyst_state

gr.close_all()

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 AI Employee Training System")

    initial_msg = {"role": "assistant", "content": initial_customer_greeting}

    customer_state = gr.State([initial_msg])
    analyst_state = gr.State([])

    with gr.Row():
        customer_box = gr.Chatbot(value=[initial_msg], label="Customer Persona")
        analyst_box = gr.Chatbot(label="Trainer / Analyst Coach")

    user_input = gr.Textbox(
        label="Type your response to the customer here...",
        placeholder="Hello! How can I help you today?",
    )
    submit_btn = gr.Button("Send to Customer")

    io_config = {
        "fn": multi_agent_chat,
        "inputs": [
            user_input,
            customer_box,
            analyst_box,
            customer_state,
            analyst_state,
        ],
        "outputs": [
            user_input,
            customer_box,
            analyst_box,
            customer_state,
            analyst_state,
        ],
    }

    submit_btn.click(**io_config)
    user_input.submit(**io_config)

demo.launch()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
